# AI4I 2020 Predictive Maintenance — Advanced Ensemble
**Target:** F1 > 0.95 · AUROC > 0.99  
**Stack:** XGBoost + LightGBM + CatBoost · Optuna (50+ trials each) · SMOTE-ENN · 10-fold CV · Youden-J threshold · Multi-target failure modes · Boruta RFE

In [ ]:
# ── 0. INSTALL DEPENDENCIES ────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

try:
    import optuna
except ImportError:
    pip_install("optuna")

try:
    from imblearn.combine import SMOTEENN
except ImportError:
    pip_install("imbalanced-learn")

try:
    from catboost import CatBoostClassifier
except ImportError:
    pip_install("catboost")

try:
    import boruta
except ImportError:
    pip_install("boruta")

print("All dependencies ready.")

In [ ]:
# ── 1. IMPORTS ─────────────────────────────────────────────────────────────
import warnings, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_recall_fscore_support,
    accuracy_score, precision_recall_curve, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

try:
    from boruta import BorutaPy
    BORUTA_AVAILABLE = True
except ImportError:
    BORUTA_AVAILABLE = False

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print(f"Output dir: {OUT}")

In [ ]:
# ── 2. LOAD DATA ───────────────────────────────────────────────────────────
data_path = None
for base in [Path("/kaggle/input"), Path("./data"), Path(".")]:
    if not base.exists():
        continue
    for f in base.rglob("ai4i2020.csv"):
        data_path = f
        break
    if data_path:
        break

if data_path is None:
    raise FileNotFoundError("ai4i2020.csv not found — add the dataset source in kernel-metadata.json")

print(f"Loading: {data_path}")
raw = pd.read_csv(data_path)
print(f"Shape: {raw.shape}")
print(raw.dtypes)
raw.head(3)

In [ ]:
# ── 3. BASIC CLEANING & RENAMING ───────────────────────────────────────────
df = raw.copy()

# Drop non-informative ID columns
df = df.drop(columns=[c for c in ["UDI", "Product ID"] if c in df.columns])

# Identify multi-target failure mode columns (before renaming)
failure_mode_raw = [c for c in df.columns if "Failure Type" in c or "failure" in c.lower()]
print("Potential failure columns:", failure_mode_raw)

# Standardise column names
rename_map = {
    "Air temperature [K]":       "air_temp_k",
    "Process temperature [K]":   "process_temp_k",
    "Rotational speed [rpm]":    "rotational_speed_rpm",
    "Torque [Nm]":               "torque_nm",
    "Tool wear [min]":           "tool_wear_min",
    "Machine failure":           "fail",
    "TWF":                       "twf",   # Tool Wear Failure
    "HDF":                       "hdf",   # Heat Dissipation Failure
    "PWF":                       "pwf",   # Power Failure
    "OSF":                       "osf",   # Overstrain Failure
    "RNF":                       "rnf",   # Random Failure
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# One-hot encode machine type
if "Type" in df.columns:
    df = pd.get_dummies(df, columns=["Type"], drop_first=True)

# Identify failure mode columns after rename
FAILURE_MODES = [c for c in ["twf", "hdf", "pwf", "osf", "rnf"] if c in df.columns]
print(f"Failure mode targets: {FAILURE_MODES}")
print(f"Primary target 'fail' positive rate: {df['fail'].mean():.4f}")
df.head(3)

In [ ]:
# ── 4. FEATURE ENGINEERING ─────────────────────────────────────────────────
#
# 4a. Domain-specific physics features
df["temp_diff"]          = df["process_temp_k"] - df["air_temp_k"]
df["power_w"]            = df["torque_nm"] * df["rotational_speed_rpm"] * (2.0 * np.pi / 60.0)
df["overstrain_score"]   = df["tool_wear_min"] * df["torque_nm"] / 11_000.0
df["heat_diss_flag"]     = ((df["temp_diff"] < 8.6) & (df["rotational_speed_rpm"] < 1380)).astype(np.int8)
df["power_oor"]          = ((df["power_w"] < 3_500) | (df["power_w"] > 9_000)).astype(np.int8)
df["specific_energy"]    = df["power_w"] / (df["rotational_speed_rpm"] + 1e-6)  # torque proxy
df["wear_rate"]          = df["tool_wear_min"] / (df["rotational_speed_rpm"] + 1e-6)
df["thermal_stress"]     = df["temp_diff"] * df["torque_nm"]
df["mech_power_density"] = df["power_w"] / (df["temp_diff"].abs() + 1.0)

# 4b. Interaction features (products, ratios, differences)
df["torque_x_wear"]      = df["torque_nm"]              * df["tool_wear_min"]
df["rpm_x_wear"]         = df["rotational_speed_rpm"]   * df["tool_wear_min"]
df["power_x_wear"]       = df["power_w"]                * df["tool_wear_min"]
df["temp_x_wear"]        = df["temp_diff"]              * df["tool_wear_min"]
df["torque_x_rpm"]       = df["torque_nm"]              * df["rotational_speed_rpm"]
df["torque_div_rpm"]     = df["torque_nm"]              / (df["rotational_speed_rpm"] + 1e-6)
df["temp_div_rpm"]       = df["temp_diff"]              / (df["rotational_speed_rpm"] + 1e-6)
df["wear_x_stress"]      = df["tool_wear_min"]          * df["thermal_stress"]
df["power_x_temp"]       = df["power_w"]                * df["temp_diff"]

# 4c. Polynomial features for top numeric columns (degree=2, interactions only to save RAM)
POLY_COLS = ["torque_nm", "tool_wear_min", "power_w", "overstrain_score", "temp_diff"]
for i, ci in enumerate(POLY_COLS):
    df[f"{ci}_sq"] = df[ci] ** 2
    for cj in POLY_COLS[i+1:]:
        df[f"{ci}_x_{cj}"] = df[ci] * df[cj]

# 4d. Rank/quantile features for top numeric columns (robust to outliers)
RANK_COLS = ["power_w", "torque_nm", "tool_wear_min", "overstrain_score", "temp_diff"]
for col in RANK_COLS:
    df[f"rank_{col}"] = df[col].rank(pct=True)

# 4e. Log transforms for heavy-tailed columns
for col in ["tool_wear_min", "power_w", "torque_nm", "rotational_speed_rpm"]:
    df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))

# 4f. Threshold-based risk flags (domain heuristics from AI4I paper)
df["twf_risk"]  = (df["tool_wear_min"] > 200).astype(np.int8)         # Tool Wear Failure risk
df["hdf_risk"]  = ((df["temp_diff"] < 8.6) & (df["rotational_speed_rpm"] < 1380)).astype(np.int8)
df["pwf_risk"]  = ((df["power_w"] < 3500) | (df["power_w"] > 9000)).astype(np.int8)
df["osf_risk"]  = (df["overstrain_score"] > 1.0).astype(np.int8)
df["risk_sum"]  = df[["twf_risk", "hdf_risk", "pwf_risk", "osf_risk"]].sum(axis=1)

print(f"Total columns after feature engineering: {df.shape[1]}")

In [ ]:
# ── 5. MULTI-TARGET: FAILURE MODE FEATURES ─────────────────────────────────
#
# Train a lightweight model for each failure mode; use OOF probabilities
# as additional meta-features for the main 'fail' model.

BASE_SENSOR_COLS = [
    "air_temp_k", "process_temp_k", "rotational_speed_rpm",
    "torque_nm", "tool_wear_min",
    "Type_L", "Type_M",
    "temp_diff", "power_w", "overstrain_score",
    "heat_diss_flag", "power_oor",
    "torque_x_wear", "rpm_x_wear", "power_x_wear",
]
BASE_SENSOR_COLS = [c for c in BASE_SENSOR_COLS if c in df.columns]

# Filter available type cols
type_cols = [c for c in ["Type_L", "Type_M"] if c in df.columns]

# Build full feature list (exclude target columns)
EXCLUDE = {"fail"} | set(FAILURE_MODES)
ALL_FEAT_COLS = [c for c in df.columns if c not in EXCLUDE]

X_full = df[ALL_FEAT_COLS].copy().fillna(0).astype(np.float32)
y_main = df["fail"].astype(int).values

# Per-mode OOF probabilities (10-fold, simple LightGBM)
SKF10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

mode_oof = {}
if FAILURE_MODES:
    print("Training per-failure-mode LightGBM for meta-features...")
    for mode in FAILURE_MODES:
        y_mode = df[mode].astype(int).values
        if y_mode.sum() < 5:
            print(f"  Skipping {mode}: too few positives ({y_mode.sum()})")
            continue
        oof_m = np.zeros(len(y_mode), dtype=np.float32)
        for tr_idx, te_idx in SKF10.split(X_full.values, y_mode):
            X_tr, X_te = X_full.values[tr_idx], X_full.values[te_idx]
            y_tr = y_mode[tr_idx]
            spw_m = max((y_tr == 0).sum(), 1) / max((y_tr == 1).sum(), 1)
            clf_m = lgb.LGBMClassifier(
                n_estimators=300, max_depth=5, learning_rate=0.05,
                scale_pos_weight=spw_m, num_leaves=31,
                n_jobs=-1, random_state=SEED, verbosity=-1
            )
            clf_m.fit(X_tr, y_tr)
            oof_m[te_idx] = clf_m.predict_proba(X_te)[:, 1]
        mode_oof[mode] = oof_m
        auc_m = roc_auc_score(y_mode, oof_m)
        print(f"  {mode.upper()} OOF AUROC: {auc_m:.4f}  positives: {y_mode.sum()}")
    
    # Append mode OOF probabilities to feature matrix
    for mode, oof_arr in mode_oof.items():
        X_full[f"oof_{mode}"] = oof_arr
    print(f"Added {len(mode_oof)} failure-mode meta-features. X shape: {X_full.shape}")
else:
    print("No failure mode columns found — skipping multi-target step.")

In [ ]:
# ── 6. BORUTA / RFE FEATURE SELECTION ──────────────────────────────────────

X_arr_full = X_full.values.astype(np.float32)
feat_names = list(X_full.columns)

selected_features = feat_names  # default: keep all

if BORUTA_AVAILABLE:
    print("Running Boruta feature selection (max_iter=100)...")
    rf_for_boruta = RandomForestClassifier(
        n_estimators=200, max_depth=7, class_weight="balanced",
        n_jobs=-1, random_state=SEED
    )
    boruta_sel = BorutaPy(
        rf_for_boruta, n_estimators="auto", max_iter=100,
        alpha=0.05, random_state=SEED, verbose=0
    )
    boruta_sel.fit(X_arr_full, y_main)
    selected_mask = boruta_sel.support_ | boruta_sel.support_weak_
    selected_features = [feat_names[i] for i, m in enumerate(selected_mask) if m]
    print(f"Boruta selected {len(selected_features)} / {len(feat_names)} features")
else:
    # Fallback: Random Forest permutation importance RFE
    print("Boruta unavailable — using RF permutation importance RFE...")
    rf_imp = RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced",
        n_jobs=-1, random_state=SEED
    )
    rf_imp.fit(X_arr_full, y_main)
    imp = rf_imp.feature_importances_
    # Keep top 80% by cumulative importance
    sorted_idx = np.argsort(imp)[::-1]
    cumsum = np.cumsum(imp[sorted_idx]) / imp.sum()
    keep_n = int(np.searchsorted(cumsum, 0.99)) + 1
    selected_features = [feat_names[i] for i in sorted_idx[:keep_n]]
    print(f"RFE selected top {len(selected_features)} / {len(feat_names)} features (99% importance)")

# Subset feature matrix to selected features
X_sel = X_full[selected_features].values.astype(np.float32)
print(f"Final feature matrix shape: {X_sel.shape}")

In [ ]:
# ── 7. CLASS IMBALANCE: SMOTE-ENN ──────────────────────────────────────────
#
# We apply SMOTE-ENN INSIDE the CV loop per fold.
# Here we just define the resampler factory and a helper.

pos_count  = y_main.sum()
neg_count  = len(y_main) - pos_count
SPW        = neg_count / max(pos_count, 1)   # scale_pos_weight for cost-sensitive models

print(f"Positives: {pos_count}  Negatives: {neg_count}  scale_pos_weight: {SPW:.2f}")

def smoteenn_resample(X_tr, y_tr, seed=SEED):
    """Apply SMOTE-ENN to a training fold. Falls back to SMOTE on failure."""
    try:
        resampler = SMOTEENN(
            smote=SMOTE(k_neighbors=5, random_state=seed),
            enn=EditedNearestNeighbours(n_neighbors=3, kind_sel="all"),
            random_state=seed
        )
        X_r, y_r = resampler.fit_resample(X_tr, y_tr)
    except Exception as e:
        print(f"    SMOTE-ENN failed ({e}), falling back to SMOTE")
        resampler = SMOTE(k_neighbors=min(5, y_tr.sum()-1), random_state=seed)
        X_r, y_r = resampler.fit_resample(X_tr, y_tr)
    return X_r.astype(np.float32), y_r

print("SMOTE-ENN resampler ready.")

In [ ]:
# ── 8. OPTUNA HYPERPARAMETER TUNING ────────────────────────────────────────
#
# We use a 3-fold inner CV on a 60% subsample for speed,
# then validate on holdout. 50 trials per model.

N_OPTUNA_TRIALS = 100   # >= 50 as required
N_OPTUNA_FOLDS  = 3
OPTUNA_SEED     = SEED

# Subsample for Optuna (speeds up search, keeps class ratio)
rng_opt = np.random.default_rng(SEED)
n_sub = min(len(y_main), 8000)
sub_idx = rng_opt.choice(len(y_main), size=n_sub, replace=False)
# Ensure we have positives
pos_idx_all = np.where(y_main == 1)[0]
sub_idx = np.union1d(sub_idx, pos_idx_all)  # always include all positives
X_sub = X_sel[sub_idx]
y_sub = y_main[sub_idx]
spw_sub = (y_sub == 0).sum() / max((y_sub == 1).sum(), 1)
print(f"Optuna subsample: {len(y_sub)} rows  pos_rate: {y_sub.mean():.4f}")

inner_skf = StratifiedKFold(n_splits=N_OPTUNA_FOLDS, shuffle=True, random_state=SEED)

# ── 8a. XGBoost Optuna ─────────────────────────────────────────────────────
def xgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 400, 2000, step=100),
        max_depth        = trial.suggest_int("max_depth", 3, 9),
        learning_rate    = trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        subsample        = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
        gamma            = trial.suggest_float("gamma", 0.0, 2.0),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        scale_pos_weight = spw_sub,
        tree_method      = "hist",
        eval_metric      = "logloss",
        n_jobs           = -1,
        random_state     = SEED,
    )
    aucs = []
    for tr_i, te_i in inner_skf.split(X_sub, y_sub):
        X_tr_o, y_tr_o = smoteenn_resample(X_sub[tr_i], y_sub[tr_i], seed=SEED)
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr_o, y_tr_o, verbose=False)
        aucs.append(roc_auc_score(y_sub[te_i], clf.predict_proba(X_sub[te_i])[:, 1]))
    return np.mean(aucs)

print(f"Tuning XGBoost ({N_OPTUNA_TRIALS} trials)...")
t0 = time.time()
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
best_xgb = study_xgb.best_params
best_xgb["scale_pos_weight"] = SPW  # use full-dataset ratio for final model
best_xgb.update({"tree_method": "hist", "eval_metric": "logloss",
                 "n_jobs": -1, "random_state": SEED})
print(f"  Best XGB AUROC: {study_xgb.best_value:.4f}  ({time.time()-t0:.0f}s)")
print(f"  Params: {best_xgb}")

In [ ]:
# ── 8b. LightGBM Optuna ────────────────────────────────────────────────────
def lgb_objective(trial):
    params = dict(
        n_estimators       = trial.suggest_int("n_estimators", 400, 2000, step=100),
        max_depth          = trial.suggest_int("max_depth", 3, 10),
        num_leaves         = trial.suggest_int("num_leaves", 15, 127),
        learning_rate      = trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        subsample          = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree   = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_samples  = trial.suggest_int("min_child_samples", 5, 50),
        reg_alpha          = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda         = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        subsample_freq     = 1,
        class_weight       = "balanced",
        n_jobs             = -1,
        random_state       = SEED,
        verbosity          = -1,
    )
    aucs = []
    for tr_i, te_i in inner_skf.split(X_sub, y_sub):
        X_tr_o, y_tr_o = smoteenn_resample(X_sub[tr_i], y_sub[tr_i], seed=SEED)
        clf = lgb.LGBMClassifier(**params)
        clf.fit(X_tr_o, y_tr_o)
        aucs.append(roc_auc_score(y_sub[te_i], clf.predict_proba(X_sub[te_i])[:, 1]))
    return np.mean(aucs)

print(f"Tuning LightGBM ({N_OPTUNA_TRIALS} trials)...")
t0 = time.time()
study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
best_lgb = study_lgb.best_params
best_lgb.update({"class_weight": "balanced", "n_jobs": -1,
                 "random_state": SEED, "verbosity": -1})
print(f"  Best LGB AUROC: {study_lgb.best_value:.4f}  ({time.time()-t0:.0f}s)")
print(f"  Params: {best_lgb}")

In [ ]:
# ── 8c. CatBoost Optuna ────────────────────────────────────────────────────
def cat_objective(trial):
    params = dict(
        iterations          = trial.suggest_int("iterations", 400, 1500, step=100),
        depth               = trial.suggest_int("depth", 4, 10),
        learning_rate       = trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        l2_leaf_reg         = trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        random_strength     = trial.suggest_float("random_strength", 0.5, 5.0),
        border_count        = trial.suggest_int("border_count", 32, 254),
        subsample           = trial.suggest_float("subsample", 0.5, 1.0),
        bootstrap_type      = "Bernoulli",
        min_data_in_leaf    = trial.suggest_int("min_data_in_leaf", 1, 20),
        auto_class_weights  = "Balanced",
        random_seed         = SEED,
        verbose             = 0,
    )
    aucs = []
    for tr_i, te_i in inner_skf.split(X_sub, y_sub):
        X_tr_o, y_tr_o = smoteenn_resample(X_sub[tr_i], y_sub[tr_i], seed=SEED)
        clf = CatBoostClassifier(**params)
        clf.fit(X_tr_o, y_tr_o)
        aucs.append(roc_auc_score(y_sub[te_i], clf.predict_proba(X_sub[te_i])[:, 1]))
    return np.mean(aucs)

print(f"Tuning CatBoost ({N_OPTUNA_TRIALS} trials)...")
t0 = time.time()
study_cat = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_cat.optimize(cat_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
best_cat = study_cat.best_params
best_cat.update({"auto_class_weights": "Balanced",
                 "random_seed": SEED, "verbose": 0})
print(f"  Best CAT AUROC: {study_cat.best_value:.4f}  ({time.time()-t0:.0f}s)")
print(f"  Params: {best_cat}")

In [ ]:
# ── 9. 10-FOLD STRATIFIED CV WITH TUNED MODELS ─────────────────────────────

oof_xgb = np.zeros(len(y_main), dtype=np.float64)
oof_lgb = np.zeros(len(y_main), dtype=np.float64)
oof_cat = np.zeros(len(y_main), dtype=np.float64)

fold_results = []
t_cv_start = time.time()

print("10-fold stratified CV with tuned ensemble...")
for fold, (tr_idx, te_idx) in enumerate(SKF10.split(X_sel, y_main), 1):
    X_tr_raw, X_te = X_sel[tr_idx], X_sel[te_idx]
    y_tr_raw, y_te = y_main[tr_idx], y_main[te_idx]

    # SMOTE-ENN on training fold
    X_tr, y_tr = smoteenn_resample(X_tr_raw, y_tr_raw, seed=SEED + fold)

    # XGBoost (with early stopping)
    xgb_params = {**best_xgb, "early_stopping_rounds": 50}
    clf_x = xgb.XGBClassifier(**xgb_params)
    clf_x.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    oof_xgb[te_idx] = clf_x.predict_proba(X_te)[:, 1]

    # LightGBM (with early stopping)
    lgb_params = {**best_lgb}
    clf_l = lgb.LGBMClassifier(**lgb_params)
    clf_l.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[te_idx] = clf_l.predict_proba(X_te)[:, 1]

    # CatBoost
    clf_c = CatBoostClassifier(**best_cat)
    clf_c.fit(X_tr, y_tr)
    oof_cat[te_idx] = clf_c.predict_proba(X_te)[:, 1]

    # Per-fold blend (simple average)
    fold_blend = (oof_xgb[te_idx] + oof_lgb[te_idx] + oof_cat[te_idx]) / 3.0
    fauc = roc_auc_score(y_te, fold_blend)
    fold_results.append(fauc)
    print(f"  Fold {fold:02d}: blend AUROC={fauc:.4f}  "
          f"(XGB={roc_auc_score(y_te, oof_xgb[te_idx]):.4f}  "
          f"LGB={roc_auc_score(y_te, oof_lgb[te_idx]):.4f}  "
          f"CAT={roc_auc_score(y_te, oof_cat[te_idx]):.4f})")

cv_time = time.time() - t_cv_start
print(f"\nCV done in {cv_time:.0f}s  |  mean AUROC: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")

In [ ]:
# ── 10. STACKING META-LEARNER (XGBoost) ─────────────────────────────────────

# Build stacking features: base OOFs + failure mode OOFs
stack_cols = [oof_xgb, oof_lgb, oof_cat]
for mode in sorted(mode_oof.keys()):
    stack_cols.append(mode_oof[mode])
oof_stack = np.column_stack(stack_cols)
print(f"Stacking matrix shape: {oof_stack.shape}")

# XGBoost meta-learner with OOF to avoid leakage
from sklearn.model_selection import cross_val_predict
meta_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=2, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=SPW, tree_method="hist",
    eval_metric="logloss", random_state=SEED, n_jobs=-1
)
oof_meta = cross_val_predict(
    meta_xgb, oof_stack, y_main,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    method="predict_proba"
)[:, 1]
print(f"Meta-XGB OOF AUROC: {roc_auc_score(y_main, oof_meta):.4f}")

# ── Blending: rank-average of base models + meta ─────────────────────────
def rank_avg(*arrs):
    from scipy.stats import rankdata
    ranked = [rankdata(a) / len(a) for a in arrs]
    return np.mean(ranked, axis=0)

try:
    from scipy.stats import rankdata
    proba_blend = rank_avg(oof_xgb, oof_lgb, oof_cat, oof_meta)
except ImportError:
    proba_blend = (oof_xgb + oof_lgb + oof_cat + oof_meta) / 4.0

print(f"Blend OOF AUROC: {roc_auc_score(y_main, proba_blend):.4f}")

In [ ]:
# ── 11. THRESHOLD OPTIMIZATION (Youden's J statistic) ─────────────────────
#
# Youden's J = Sensitivity + Specificity - 1 = TPR - FPR
# This is equivalent to maximising balanced accuracy.
# We also report F1-optimal threshold for comparison.

fpr, tpr, roc_thr = roc_curve(y_main, proba_blend)
youden_j           = tpr - fpr
youden_idx         = int(np.nanargmax(youden_j))
thr_youden         = float(roc_thr[youden_idx])

prec_c, rec_c, pr_thr = precision_recall_curve(y_main, proba_blend)
f1_curve               = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-9)
f1_idx                 = int(np.nanargmax(f1_curve[:-1]))
thr_f1                 = float(pr_thr[f1_idx])

# Use Youden's J threshold as primary; compare with F1-optimal
pred_youden = (proba_blend >= thr_youden).astype(int)
pred_f1opt  = (proba_blend >= thr_f1).astype(int)

def eval_pred(y_true, y_pred, label):
    auc_val = roc_auc_score(y_true, proba_blend)
    f1_val  = f1_score(y_true, y_pred)
    acc_val = accuracy_score(y_true, y_pred)
    p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    print(f"[{label}]  AUC={auc_val:.4f}  F1={f1_val:.4f}  Acc={acc_val:.4f}  P={p:.4f}  R={r:.4f}")
    return auc_val, f1_val, acc_val, float(p), float(r)

print("\nThreshold comparison (OOF):")
auc_yj, f1_yj, acc_yj, prec_yj, rec_yj = eval_pred(y_main, pred_youden, f"Youden J (thr={thr_youden:.4f})")
auc_f1, f1_f1, acc_f1, prec_f1, rec_f1 = eval_pred(y_main, pred_f1opt,  f"F1-opt  (thr={thr_f1:.4f})")

# Pick the threshold that yields higher F1
if f1_yj >= f1_f1:
    best_thr, y_pred_final = thr_youden, pred_youden
    best_auc, best_f1, best_acc = auc_yj, f1_yj, acc_yj
    best_prec, best_rec = prec_yj, rec_yj
    thr_method = "Youden-J"
else:
    best_thr, y_pred_final = thr_f1, pred_f1opt
    best_auc, best_f1, best_acc = auc_f1, f1_f1, acc_f1
    best_prec, best_rec = prec_f1, rec_f1
    thr_method = "F1-optimal"

print(f"\nChosen threshold method: {thr_method} (thr={best_thr:.4f})")
print(f"  Final AUROC : {best_auc:.4f}")
print(f"  Final F1    : {best_f1:.4f}")
print(f"  Final Acc   : {best_acc:.4f}")

In [ ]:
# ── 12. SAVE METRICS.JSON ──────────────────────────────────────────────────

individual_aucs = {
    "auc_xgb":  round(roc_auc_score(y_main, oof_xgb), 4),
    "auc_lgb":  round(roc_auc_score(y_main, oof_lgb), 4),
    "auc_cat":  round(roc_auc_score(y_main, oof_cat), 4),
    "auc_meta": round(roc_auc_score(y_main, oof_meta), 4),
    "auc_blend": round(best_auc, 4),
}

cv_stats = {
    "cv_folds":    10,
    "cv_auc_mean": round(float(np.mean(fold_results)), 4),
    "cv_auc_std":  round(float(np.std(fold_results)), 4),
    "cv_auc_min":  round(float(np.min(fold_results)), 4),
    "cv_auc_max":  round(float(np.max(fold_results)), 4),
}

metrics = {
    "roc_auc":          round(best_auc, 4),
    "f1":               round(best_f1, 4),
    "accuracy":         round(best_acc, 4),
    "precision":        round(best_prec, 4),
    "recall":           round(best_rec, 4),
    "threshold":        round(best_thr, 4),
    "threshold_method": thr_method,
    "n_features":       len(selected_features),
    "n_original_feat":  len(feat_names),
    "train_time_s":     round(cv_time, 1),
    "optuna_trials_per_model": N_OPTUNA_TRIALS,
    **individual_aucs,
    **cv_stats,
}

with open(OUT / "metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

print("metrics.json saved:")
print(json.dumps(metrics, indent=2))

In [ ]:
# ── 13. TRAINING RESULTS VISUALISATION → training_results.png ──────────────

from sklearn.metrics import ConfusionMatrixDisplay

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {"xgb": "#e74c3c", "lgb": "#2980b9", "cat": "#27ae60", "blend": "#8e44ad"}

# ── 13a. ROC curves ────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[0, 0])
for name, proba, color in [
    ("XGBoost",  oof_xgb,    COLORS["xgb"]),
    ("LightGBM", oof_lgb,    COLORS["lgb"]),
    ("CatBoost", oof_cat,    COLORS["cat"]),
    ("Blend",    proba_blend, COLORS["blend"]),
]:
    fpr_c, tpr_c, _ = roc_curve(y_main, proba)
    auc_c = roc_auc_score(y_main, proba)
    ax_roc.plot(fpr_c, tpr_c, label=f"{name} ({auc_c:.4f})", color=color, lw=2)
ax_roc.plot([0,1], [0,1], "k--", lw=1)
ax_roc.set(xlabel="FPR", ylabel="TPR", title="ROC Curves (OOF)")
ax_roc.legend(fontsize=8)
ax_roc.grid(True, alpha=0.3)

# ── 13b. Precision-Recall curves ───────────────────────────────────────────
ax_pr = fig.add_subplot(gs[0, 1])
for name, proba, color in [
    ("XGBoost",  oof_xgb,    COLORS["xgb"]),
    ("LightGBM", oof_lgb,    COLORS["lgb"]),
    ("CatBoost", oof_cat,    COLORS["cat"]),
    ("Blend",    proba_blend, COLORS["blend"]),
]:
    p_c, r_c, _ = precision_recall_curve(y_main, proba)
    f1_avg = 2 * p_c * r_c / (p_c + r_c + 1e-9)
    ax_pr.plot(r_c, p_c, label=f"{name} (F1={f1_avg.max():.3f})", color=color, lw=2)
ax_pr.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curves (OOF)")
ax_pr.legend(fontsize=8)
ax_pr.grid(True, alpha=0.3)

# ── 13c. Confusion matrix (best blend + best threshold) ────────────────────
ax_cm = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_main, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Failure", "Failure"])
disp.plot(ax=ax_cm, colorbar=False, cmap="Blues")
ax_cm.set_title(f"Confusion Matrix\n(thr={best_thr:.3f}, {thr_method})")

# ── 13d. 10-fold CV AUROC bar chart ────────────────────────────────────────
ax_cv = fig.add_subplot(gs[1, 0])
folds = list(range(1, len(fold_results)+1))
bars = ax_cv.bar(folds, fold_results, color=COLORS["blend"], alpha=0.8, edgecolor="white")
ax_cv.axhline(np.mean(fold_results), color="red", lw=2, linestyle="--",
              label=f"Mean={np.mean(fold_results):.4f}")
ax_cv.set(xlabel="Fold", ylabel="AUROC", title="10-Fold CV AUROC", ylim=(0.95, 1.01))
ax_cv.legend(fontsize=8)
ax_cv.grid(True, alpha=0.3, axis="y")
for bar, v in zip(bars, fold_results):
    ax_cv.text(bar.get_x() + bar.get_width()/2, v + 0.0005, f"{v:.3f}",
               ha="center", va="bottom", fontsize=7)

# ── 13e. Optuna study convergence ──────────────────────────────────────────
ax_opt = fig.add_subplot(gs[1, 1])
for study, name, color in [(study_xgb, "XGB", COLORS["xgb"]),
                            (study_lgb, "LGB", COLORS["lgb"]),
                            (study_cat, "CAT", COLORS["cat"])]:
    vals = [t.value for t in study.trials if t.value is not None]
    best_so_far = pd.Series(vals).cummax().values
    ax_opt.plot(range(1, len(best_so_far)+1), best_so_far, label=name, color=color, lw=2)
ax_opt.set(xlabel="Trial", ylabel="Best AUROC", title="Optuna Convergence")
ax_opt.legend(fontsize=8)
ax_opt.grid(True, alpha=0.3)

# ── 13f. Probability distributions ────────────────────────────────────────
ax_dist = fig.add_subplot(gs[1, 2])
ax_dist.hist(proba_blend[y_main==0], bins=50, alpha=0.6, color="steelblue", label="No Failure", density=True)
ax_dist.hist(proba_blend[y_main==1], bins=50, alpha=0.6, color="tomato",    label="Failure",    density=True)
ax_dist.axvline(best_thr, color="black", lw=2, linestyle="--", label=f"thr={best_thr:.3f}")
ax_dist.set(xlabel="Predicted Probability", ylabel="Density",
            title="Blend Probability Distribution")
ax_dist.legend(fontsize=8)
ax_dist.grid(True, alpha=0.3)

# ── 13g. Feature importance (top 20) ───────────────────────────────────────
ax_fi = fig.add_subplot(gs[2, :])

# Re-train one LGB on full data to get importances
clf_fi = lgb.LGBMClassifier(**best_lgb)
clf_fi.fit(X_sel, y_main)
fi_vals  = clf_fi.feature_importances_
fi_pairs = sorted(zip(selected_features, fi_vals), key=lambda x: x[1], reverse=True)[:20]
fi_names = [p[0] for p in fi_pairs]
fi_imps  = [p[1] for p in fi_pairs]

ax_fi.barh(fi_names[::-1], fi_imps[::-1], color=COLORS["lgb"], alpha=0.85)
ax_fi.set(xlabel="Importance", title="Top-20 Feature Importances (LightGBM, full-data fit)")
ax_fi.grid(True, alpha=0.3, axis="x")

# ── Title & save ───────────────────────────────────────────────────────────
fig.suptitle(
    f"AI4I Predictive Maintenance — Advanced Ensemble\n"
    f"AUROC={best_auc:.4f}  F1={best_f1:.4f}  Acc={best_acc:.4f}  "
    f"Threshold={best_thr:.4f} ({thr_method})",
    fontsize=13, fontweight="bold", y=1.01
)

out_png = OUT / "training_results.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"training_results.png saved → {out_png}")

In [ ]:
# ── 14. FINAL SUMMARY ──────────────────────────────────────────────────────

print("=" * 60)
print("AI4I PREDICTIVE MAINTENANCE — FINAL RESULTS")
print("=" * 60)
print(f"  AUROC          : {best_auc:.4f}   (target > 0.99)")
print(f"  F1 Score       : {best_f1:.4f}   (target > 0.95)")
print(f"  Accuracy       : {best_acc:.4f}")
print(f"  Precision      : {best_prec:.4f}")
print(f"  Recall         : {best_rec:.4f}")
print(f"  Threshold      : {best_thr:.4f} ({thr_method})")
print("-" * 60)
print(f"  Individual AUROC:")
print(f"    XGBoost  : {individual_aucs['auc_xgb']:.4f}")
print(f"    LightGBM : {individual_aucs['auc_lgb']:.4f}")
print(f"    CatBoost : {individual_aucs['auc_cat']:.4f}")
print(f"    Meta-LR  : {individual_aucs['auc_meta']:.4f}")
print("-" * 60)
print(f"  10-fold CV AUROC: {cv_stats['cv_auc_mean']:.4f} ± {cv_stats['cv_auc_std']:.4f}")
print(f"  Features used  : {len(selected_features)} / {len(feat_names)}")
print(f"  Optuna trials  : {N_OPTUNA_TRIALS} per model")
print(f"  CV time        : {cv_time:.0f}s")
print("=" * 60)

target_f1_met  = best_f1  > 0.95
target_auc_met = best_auc > 0.99
print(f"\n  F1  > 0.95  : {'ACHIEVED' if target_f1_met  else 'NOT MET — consider threshold tuning'}")
print(f"  AUC > 0.99  : {'ACHIEVED' if target_auc_met else 'NOT MET — ensemble near target'}")